# Robust MPC Workbench

This notebook is a single place to:

- run the MuJoCo simulation interactively
- perform quick sanity tests after code changes
- keep track of future experiments and improvements


## Environment setup

Run this first so the notebook can import the refactored `src/` package from the repository root.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.simulation import SimulationConfig, run_simulation

PROJECT_ROOT

## Quick simulation run

This cell runs a short headless simulation and plots the joint trajectories directly in the notebook.

In [ ]:
quick_config = SimulationConfig(
    duration=0.2,
    render_video=False,
    save_plots=False,
    output_dir=PROJECT_ROOT / "artifacts" / "notebook-quick-run",
)

quick_result = run_simulation(quick_config)

print(f"Steps: {len(quick_result.time_steps)}")
print(f"Infeasible steps: {int(quick_result.infeasible_steps.sum())}")
print(f"Final position error norm: {np.linalg.norm(quick_result.joint_positions[-1] - quick_result.target_positions):.4f}")

plt.figure(figsize=(10, 5))
for joint_index in range(quick_result.joint_positions.shape[1]):
    plt.plot(
        quick_result.time_steps,
        quick_result.joint_positions[:, joint_index],
        label=f"Joint {joint_index + 1}",
    )
    plt.axhline(
        quick_result.target_positions[joint_index],
        color="black",
        linestyle="--",
        alpha=0.2,
    )

plt.title("Quick simulation trajectory")
plt.xlabel("Time (s)")
plt.ylabel("Joint position (rad)")
plt.grid(True)
plt.legend(loc="upper right", ncol=2)
plt.tight_layout()
plt.show()

## Full artifact run

Use this when you want the saved video and plots. The outputs are written into a dedicated notebook artifact folder.

In [ ]:
full_config = SimulationConfig(
    duration=1.0,
    render_video=True,
    save_plots=True,
    output_dir=PROJECT_ROOT / "artifacts" / "notebook-full-run",
)

# Uncomment to generate notebook artifacts.
# full_result = run_simulation(full_config)
# full_result.video_path, full_result.plot_paths

## Sanity tests

These checks are lightweight and are meant to catch obvious regressions after refactors.

In [ ]:
def run_sanity_checks(duration: float = 0.05) -> None:
    test_config = SimulationConfig(
        duration=duration,
        render_video=False,
        save_plots=False,
        output_dir=PROJECT_ROOT / "artifacts" / "notebook-tests",
    )
    result = run_simulation(test_config)

    assert result.time_steps.ndim == 1
    assert result.joint_positions.shape[0] == result.time_steps.shape[0]
    assert result.control_inputs.shape[0] == result.time_steps.shape[0]
    assert result.joint_positions.shape[1] == result.target_positions.shape[0]
    assert result.control_inputs.shape[1] == result.target_positions.shape[0]
    assert np.isfinite(result.joint_positions).all()
    assert np.isfinite(result.control_inputs).all()
    assert np.isfinite(result.tube_bounds).all()
    assert result.infeasible_steps.shape[0] == result.time_steps.shape[0]
    assert result.time_steps.shape[0] > 0

    print("Sanity checks passed.")
    print(f"Steps: {result.time_steps.shape[0]}")
    print(f"Max abs control: {np.abs(result.control_inputs).max():.4f}")
    print(f"Infeasible steps: {int(result.infeasible_steps.sum())}")


run_sanity_checks()

## Experiment ideas

Try editing these parameters during exploration:

- `horizon` to compare short versus long planning windows
- `disturbance_bound` and `tube_disturbance_bound` to study robustness margins
- `position_weight`, `velocity_weight`, and `control_weight` to tune tracking versus aggressiveness
- `target_positions` for different reach goals
- `initial_positions` for off-nominal starting states


## Future work

- Add benchmark cells that compare multiple controller settings in one notebook run.
- Track summary metrics such as settling time, peak control magnitude, and final error norm.
- Add disturbance sweep experiments and save the results as a table for later analysis.
- Explore a richer plant model than the double-integrator approximation used by the MPC.
- Add notebook cells for comparing unconstrained, constrained, and tube-based MPC variants.
- Move repeated experiment code into reusable helper functions if the notebook grows.
